In [21]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import sum, avg,col,first,when,udf,regexp_replace,to_timestamp
from pyspark.sql.types import StringType, DoubleType, ArrayType
import re
ss = SparkSession.builder.config("spark.jars", "/home/jrodarte/postgresql-42.7.3.jar").getOrCreate()

In [22]:
dfMovimientos = ss.read.format("csv").options(header='true', inferSchema='true', delimiter=',').load("documentos/movimientos0725.csv")
#dfMovimientos.schema

In [23]:
# Datos de conexión
user = "jrodarte"
password = "roma1993_"

dfTipoMovimientos = ss.read.format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "(select * from prestadero.tipomovimientos)") \
    .option("user", user) \
    .option("password", password) \
    .option("driver", "org.postgresql.Driver") \
    .load()

#dfTipoMovimientos.show()

In [25]:
def limpiarImporte(importe):
    return importe.replace("$", "").replace(",", "")


limpiarImpoUDF = udf(limpiarImporte, StringType())

In [26]:
nuevosNombresMov = ['autorizacion', 'feoperacion', 'tipo', 'movimiento', 'importe', 'estatus', 'referencia']
nuevosNombresDetMov = ['autorizacion', 'detalle']
dfDetalleMovimiento = dfMovimientos.select('Autorización', 'Detalle')
dfMovimientos = dfMovimientos.selectExpr(
    "`Autorización`",
    "`Fecha operación`",
    "`Tipo`",
    "`Movimiento`",
    "`Importe`",
    "`Estatus`",
    "`Ref. 1`"
)

In [27]:
dfMovimientos = dfMovimientos.toDF(*nuevosNombresMov)
dfMovimientos = dfMovimientos.withColumn('feoperacion', to_timestamp(col("feoperacion"), "dd/MM/yyyy HH:mm:ss"))
dfMovimientos = dfMovimientos.withColumn('importeS', limpiarImpoUDF(col("importe")))
dfMovimientos = dfMovimientos.withColumn('importe', col("importeS").cast(DoubleType())).drop("importeS")
dfDetalleMovimiento = dfDetalleMovimiento.toDF(*nuevosNombresDetMov)
#dfMovimientos.show()
#dfMovimientos.schema

In [28]:
dfMovimientos = dfMovimientos.join(dfTipoMovimientos, dfMovimientos.movimiento == dfTipoMovimientos.tipomovimiento, "inner")
dfMovimientos = dfMovimientos.drop('tipomovimiento').drop('movimiento')
#dfMovimientos.schema

In [20]:
dfMovimientos.write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://localhost:5432/postgres") \
    .option("dbtable", "prestadero.movimientos") \
    .option("user", "jrodarte") \
    .option("password", "roma1993_") \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [10]:
dfDetalleMovimiento.show()

+------------+--------------------+
|autorizacion|             detalle|
+------------+--------------------+
|   104924819|                NULL|
|   104924820|                NULL|
|   104924821|Principal:4.04683...|
|   104923175|                NULL|
|   104923176|                NULL|
|   104923177|Principal:3.09670...|
|   104859707|Principal:0.00000...|
|   104732052|                NULL|
|   104732053|                NULL|
|   104732054|Principal:10.4385...|
|   104673705|                NULL|
|   104673706|                NULL|
|   104673707|Principal:0.00000...|
|   104636402|Principal:10.6304...|
|   104635868|Principal:5.31522...|
|   104634793|Principal:8.68985...|
|   104579832|Principal:12.4996...|
|   104551730|Principal:17.5966...|
|   104536750|Principal:24.0733...|
|   104530273|Principal:17.8160...|
+------------+--------------------+
only showing top 20 rows


In [11]:
ptrDigitos = r"\d\s"
ptrEspacioFinal = r"\s$"
def limpiezaDetalleMov(detalleMov):
    if detalleMov is None:
        return []
    else:
        coincidencia = re.findall(ptrDigitos, detalleMov)
        lenCoincidencia = len(coincidencia)
        arrDetalle = re.split(ptrDigitos, detalleMov)
        
        return arrDetalle

limpiezaDetalleMovUDF = udf(limpiezaDetalleMov, ArrayType(StringType()))

In [12]:
dfDetalleMovimiento = dfDetalleMovimiento.withColumn('arrDetalle', limpiezaDetalleMovUDF(dfDetalleMovimiento.detalle))
dfDetalleMovimiento.show()
#dfDetalleMovimiento.write.mode('overwrite').csv('Proyectos/prestadero/detalleMov.csv')

+------------+--------------------+--------------------+
|autorizacion|             detalle|          arrDetalle|
+------------+--------------------+--------------------+
|   104924819|                NULL|                  []|
|   104924820|                NULL|                  []|
|   104924821|Principal:4.04683...|[Principal:4.0468...|
|   104923175|                NULL|                  []|
|   104923176|                NULL|                  []|
|   104923177|Principal:3.09670...|[Principal:3.0967...|
|   104859707|Principal:0.00000...|[Principal:0.0000...|
|   104732052|                NULL|                  []|
|   104732053|                NULL|                  []|
|   104732054|Principal:10.4385...|[Principal:10.438...|
|   104673705|                NULL|                  []|
|   104673706|                NULL|                  []|
|   104673707|Principal:0.00000...|[Principal:0.0000...|
|   104636402|Principal:10.6304...|[Principal:10.630...|
|   104635868|Principal:5.31522

In [13]:
def procesarDetalleMov(aut,arrDetalle):
    arrDetalleNuevo = []
    for i in arrDetalle:
        Autitem = ("autorizacion", aut)
        Conitem = ("concepto", i.split(':')[0])
        Monitem = ("monto", i.split(':')[1])
        arrDetalleNuevo.append([Autitem,Conitem,Monitem])

    dictDetalleNuevo = dict(arrDetalleNuevo)
    return dictDetalleNuevo

In [227]:
procesarDetalleMov(103124028,["Principal: 0.0000", "Interes: 0.0000", "Impuesto Interes:  0.0000", "Moratorios: 0.0028",  "Impuesto Moratorios: 0.0004", "IVA Comisión Moratorios:  0.00000"])

ValueError: dictionary update sequence element #0 has length 3; 2 is required